# DSAR Intake — pull OneTrust requests into `dsar_request`

The intake step that **replaces the seeded demo requests** in `00`. It pulls open privacy requests from
**OneTrust's Privacy Rights Automation REST API** and **upserts** them (idempotent `MERGE`) into
`dsar_request` with `status='PENDING'`, so the erasure job (`05_run_erasure_job`) has real work to do.

**Shape:** OAuth2 client-credentials → paginated `GET` of the request queue → map each request to
`(request_id, first_name, last_name, email, request_type, request_date, deadline_date, status)` → `MERGE`
into `dsar_request` on `request_id`.

**Idempotent:** re-running never duplicates a request and never resets one already `COMPLETE` — `MERGE`
only inserts new IDs and refreshes still-open ones.

> **Auth is config-driven via Databricks Secrets** — no credentials in the notebook. Set
> `onetrust_base_url`, and a secret scope holding `client_id` / `client_secret`. Until Allegiant's OneTrust
> creds are wired, run with `use_mock=true` to exercise the full MERGE path against a small sample payload.

Wire this as **task 1** of the monthly Job, with `05_run_erasure_job` as **task 2** (depends on task 1).


## 0. Configuration

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema",  "allegiant_air_dsar", "2 Schema")
dbutils.widgets.text("onetrust_base_url", "https://<tenant>.onetrust.com", "3 OneTrust base URL")
dbutils.widgets.text("secret_scope", "allegiant_onetrust", "4 Databricks secret scope")
dbutils.widgets.text("deadline_days", "45", "5 Deadline = request_date + N days")
dbutils.widgets.dropdown("use_mock", "true", ["true", "false"], "6 Use mock payload (no live API call)")
dbutils.widgets.text("page_size", "200", "7 API page size")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
BASE_URL = dbutils.widgets.get("onetrust_base_url").rstrip("/")
SCOPE_NAME = dbutils.widgets.get("secret_scope")
DDAYS   = int(dbutils.widgets.get("deadline_days"))
USE_MOCK = dbutils.widgets.get("use_mock") == "true"
PAGE_SIZE = int(dbutils.widgets.get("page_size"))
FQ = f"{CATALOG}.{SCHEMA}"
print(f"Schema: {FQ} | OneTrust: {BASE_URL} | use_mock: {USE_MOCK}")


## 1. Ensure the request table exists
Same schema as `00`/`05`. Created here (if missing) so intake can run standalone in production without the
demo setup notebook. Existing data is untouched.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FQ}.dsar_request (
  request_id    STRING,
  first_name    STRING,
  last_name     STRING,
  email         STRING,
  request_type  STRING,
  request_date  DATE,
  deadline_date DATE,
  status        STRING
) COMMENT 'DSAR/CCPA privacy requests, upserted from OneTrust Privacy Rights Automation.'
""")
print("dsar_request ready")


## 2. OAuth2 token (client-credentials)
Reads `client_id` / `client_secret` from the Databricks secret scope — never hard-coded. OneTrust issues a
bearer token from `/api/access/v1/oauth/token`. Skipped entirely in mock mode.

In [ ]:
import requests

def onetrust_token():
    client_id     = dbutils.secrets.get(SCOPE_NAME, "client_id")
    client_secret = dbutils.secrets.get(SCOPE_NAME, "client_secret")
    resp = requests.post(
        f"{BASE_URL}/api/access/v1/oauth/token",
        data={"grant_type": "client_credentials",
              "client_id": client_id, "client_secret": client_secret},
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

TOKEN = None if USE_MOCK else onetrust_token()
print("token acquired" if TOKEN else "mock mode: no token")


## 3. Fetch open requests (paginated)
Pulls the request queue page by page. The exact endpoint/field names vary by OneTrust tenant/version —
confirm against Allegiant's tenant and adjust `map_request` in §4 only. In mock mode we return a small,
representative payload so the rest of the notebook runs unchanged.

In [ ]:
def fetch_live():
    """GET the OneTrust request queue, following pagination. Returns a list of raw request dicts."""
    out, page = [], 0
    headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
    while True:
        r = requests.get(
            f"{BASE_URL}/api/datasubject/v3/requestqueues",
            params={"page": page, "size": PAGE_SIZE, "status": "OPEN"},
            headers=headers, timeout=60,
        )
        r.raise_for_status()
        body = r.json()
        content = body.get("content", body if isinstance(body, list) else [])
        out.extend(content)
        # OneTrust paged responses expose `last`; fall back to short-page detection.
        if body.get("last", True) or len(content) < PAGE_SIZE:
            break
        page += 1
    return out

def fetch_mock():
    """Representative OneTrust-style payload — mixed erasure/deletion, one email-only, one with apostrophe."""
    return [
        {"requestQueueRefId": "OT-1001", "firstName": "Andra", "lastName": "Mayfield",
         "email": "andra.mayfield@example.com", "subjectType": "Customer",
         "requestType": "ERASURE", "createdDate": "2026-07-20"},
        {"requestQueueRefId": "OT-1002", "firstName": "Richard", "lastName": "Jones",
         "email": "richard.jones@example.com", "subjectType": "Customer",
         "requestType": "OPT_OUT", "createdDate": "2026-07-21"},
        {"requestQueueRefId": "OT-1003", "firstName": "James", "lastName": "O'Brien",
         "email": "james.obrien@example.com", "subjectType": "Customer",
         "requestType": "DELETION", "createdDate": "2026-07-22"},
    ]

raw = fetch_mock() if USE_MOCK else fetch_live()
print(f"fetched {len(raw)} request(s)")


## 4. Map OneTrust → `dsar_request` rows
Normalise each raw request to our schema. **`request_type` mapping** is the one policy decision: OneTrust
request types are collapsed to our two erasure modes — a hard delete/erasure → `DELETE`; anything else
that still requires PII removal (opt-out / access-then-purge) → `OBFUSCATE` (redact PII, keep the row and
its non-PII). Adjust the map to Allegiant's policy. Only rows we can act on (have an email or a full name)
are kept.

In [ ]:
from datetime import date, timedelta

# OneTrust requestType -> our erasure mode. DELETE removes the row; OBFUSCATE redacts PII in place.
TYPE_MAP = {
    "ERASURE": "DELETE", "DELETION": "DELETE", "RIGHT_TO_DELETE": "DELETE",
    "OPT_OUT": "OBFUSCATE", "DO_NOT_SELL": "OBFUSCATE", "ACCESS": "OBFUSCATE",
}
DEFAULT_MODE = "OBFUSCATE"   # safest default: redact, don't hard-delete, when the type is unknown

def g(d, *keys):
    for k in keys:
        v = d.get(k)
        if v not in (None, ""):
            return v
    return None

def map_request(rr):
    rid   = g(rr, "requestQueueRefId", "id", "requestId")
    fn    = g(rr, "firstName", "first_name") or ""
    ln    = g(rr, "lastName", "last_name") or ""
    email = g(rr, "email", "emailAddress")
    rtype = (g(rr, "requestType", "type") or "").upper()
    mode  = TYPE_MAP.get(rtype, DEFAULT_MODE)
    created = g(rr, "createdDate", "submittedDate")
    return {"request_id": rid, "first_name": fn, "last_name": ln, "email": email,
            "request_type": mode, "request_date": created}

mapped = [map_request(r) for r in raw]
# keep only actionable requests (need an email or a full name to match a subject)
actionable = [m for m in mapped if m["request_id"] and (m["email"] or (m["first_name"] and m["last_name"]))]
dropped = len(mapped) - len(actionable)
print(f"{len(actionable)} actionable request(s)" + (f"; dropped {dropped} with no identifier" if dropped else ""))


## 5. Upsert into `dsar_request` (idempotent `MERGE`)
`MERGE` on `request_id`: insert new requests as `PENDING`; refresh identifiers/type/deadline on still-open
requests; **never touch `COMPLETE` rows** (already-erased subjects stay scrubbed). Deadline is
`request_date + deadline_days` (defaults to today when the source has no created date).

In [ ]:
def sqlstr(v):
    return "NULL" if v is None else "'" + str(v).replace("'", "''") + "'"

if not actionable:
    print("nothing to upsert")
else:
    src_rows = ",\n".join(
        f"({sqlstr(m['request_id'])}, {sqlstr(m['first_name'])}, {sqlstr(m['last_name'])}, "
        f"{sqlstr(m['email'])}, {sqlstr(m['request_type'])}, "
        f"COALESCE({sqlstr(m['request_date'])}::DATE, current_date()))"
        for m in actionable
    )
    spark.sql(f"""
    MERGE INTO {FQ}.dsar_request AS t
    USING (
      SELECT * FROM (VALUES
      {src_rows}
      ) AS s(request_id, first_name, last_name, email, request_type, request_date)
    ) AS s
    ON t.request_id = s.request_id
    WHEN MATCHED AND t.status <> 'COMPLETE' THEN UPDATE SET
      t.first_name = s.first_name, t.last_name = s.last_name, t.email = s.email,
      t.request_type = s.request_type,
      t.request_date = s.request_date,
      t.deadline_date = date_add(s.request_date, {DDAYS}),
      t.status = 'PENDING'
    WHEN NOT MATCHED THEN INSERT
      (request_id, first_name, last_name, email, request_type, request_date, deadline_date, status)
      VALUES (s.request_id, s.first_name, s.last_name, s.email, s.request_type,
              s.request_date, date_add(s.request_date, {DDAYS}), 'PENDING')
    """)
    print("MERGE complete")

display(spark.sql(f"SELECT status, count(*) n FROM {FQ}.dsar_request GROUP BY status ORDER BY status"))
display(spark.sql(f"SELECT * FROM {FQ}.dsar_request WHERE status='PENDING' ORDER BY request_id"))
